# Notebook 1: Data Preparation (Standalone Code Bundle)
This notebook runs as its own Code Bundle. It:
1. Creates raw order data
2. Computes summary metrics
3. Writes output to ORDER_SUMMARY table
4. Passes output table name + row count to the next Task via `SYSTEM$SET_RETURN_VALUE`

In [ ]:
import sys
import json
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import session as snowpark_session

# Handle multiple active sessions in headless (non-interactive) mode
try:
    session = get_active_session()
except Exception:
    # Multiple sessions exist - pick the first active one
    session = next(iter(snowpark_session._active_sessions))

# Default values
TARGET_DB = 'AGENTHIVE_LINEAGE'
TARGET_SCHEMA = 'CODE_BUNDLE_DEMO'
RUN_DATE = '2026-09-30'

# Filter out kernel launcher args (only present in interactive mode)
user_args = [a for a in sys.argv[1:] if not a.startswith('-') and not a.startswith('/')]

if len(user_args) > 0:
    RUN_DATE = user_args[0]
if len(user_args) > 1:
    TARGET_DB = user_args[1]

print(f"Run date: {RUN_DATE}")
print(f"Target DB: {TARGET_DB}")
print(f"Target Schema: {TARGET_SCHEMA}")

In [ ]:
# Step 1: Create sample raw data (simulating a source table)
session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.RAW_ORDERS AS
SELECT 
  SEQ4() AS ORDER_ID,
  DATEADD('day', -MOD(SEQ4(), 30), '{RUN_DATE}'::DATE) AS ORDER_DATE,
  UNIFORM(1, 100, RANDOM()) AS CUSTOMER_ID,
  ARRAY_CONSTRUCT('Electronics','Clothing','Food','Home','Sports')[MOD(SEQ4(), 5)]::VARCHAR AS CATEGORY,
  ROUND(UNIFORM(10.00, 500.00, RANDOM())::NUMERIC(10,2), 2) AS AMOUNT
FROM TABLE(GENERATOR(ROWCOUNT => 1000))
""").collect()

print("Raw orders table created with 1000 rows.")

In [ ]:
# Step 2: Compute summary metrics by category
summary_df = session.sql(f"""
SELECT 
  CATEGORY,
  COUNT(*) AS ORDER_COUNT,
  SUM(AMOUNT) AS TOTAL_REVENUE,
  ROUND(AVG(AMOUNT), 2) AS AVG_ORDER_VALUE,
  COUNT(DISTINCT CUSTOMER_ID) AS UNIQUE_CUSTOMERS,
  '{RUN_DATE}' AS REPORT_DATE
FROM {TARGET_DB}.{TARGET_SCHEMA}.RAW_ORDERS
GROUP BY CATEGORY
ORDER BY TOTAL_REVENUE DESC
""")

summary_df.show()

In [ ]:
# Step 3: Write summary to output table
summary_df.write.mode('overwrite').save_as_table(
    f'{TARGET_DB}.{TARGET_SCHEMA}.ORDER_SUMMARY'
)

row_count = session.sql(f'SELECT COUNT(*) AS CNT FROM {TARGET_DB}.{TARGET_SCHEMA}.ORDER_SUMMARY').collect()[0]['CNT']
print(f"Summary table written with {row_count} categories.")

# Set output variables for next notebook (shared via %run)
OUTPUT_TABLE = f'{TARGET_DB}.{TARGET_SCHEMA}.ORDER_SUMMARY'
CATEGORY_COUNT = row_count
PREP_STATUS = 'SUCCESS'

print(f"OUTPUT_TABLE = {OUTPUT_TABLE}")
print(f"CATEGORY_COUNT = {CATEGORY_COUNT}")
print(f"PREP_STATUS = {PREP_STATUS}")

In [ ]:
# Pass output to next Task via SYSTEM$SET_RETURN_VALUE
# Use pipe-delimited format (not JSON) because ARGUMENTS strips double quotes
return_payload = f"{OUTPUT_TABLE}|{CATEGORY_COUNT}|{RUN_DATE}|{PREP_STATUS}"

try:
    session.sql(f"SELECT SYSTEM$SET_RETURN_VALUE('{return_payload}')").collect()
    print(f"Return value set for next task: {return_payload}")
except Exception as e:
    if 'must be called from within a task' in str(e):
        print(f"Interactive mode - skipping SYSTEM$SET_RETURN_VALUE")
        print(f"Payload: {return_payload}")
    else:
        raise